In [ ]:
%pip install yfinance pandas mysql-connector-python

### Load Stock Data into AWS RDS

##### Download daily stock data for AAPL, AMZN, GOOGL, MSFT, and TSLA for 2020–2022, save local CSV files, and load each company into an AWS RDS MySQL table
##### Query the loaded records to identify each company's highest-volume trading month

### Download Data

In [ ]:
from pathlib import Path
import yfinance as yf

# CSV files will be saved in a data folder
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

for ticker in tickers:
    stock = yf.download(
        ticker,
        start="2020-01-01",
        end="2023-01-01",
        interval="1d",
        auto_adjust=False,
        multi_level_index=False,
        progress=False
    )

    if stock.empty:
        raise ValueError(f"No data downloaded for {ticker}")

    # Move the date from the index into a regular column
    stock = stock.reset_index()

    # Keep the columns need for the assignment
    stock = stock[
        ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    ]

    stock["Date"] = stock["Date"].dt.strftime("%Y-%m-%d")

    file_path = data_folder / f"{ticker}.csv"
    stock.to_csv(file_path, index=False)

    print(
        ticker,
        "| Rows:", len(stock),
        "| First date:", stock["Date"].min(),
        "| Last date:", stock["Date"].max()
    )

In [ ]:
import pandas as pd

apple = pd.read_csv(data_folder / "AAPL.csv")

print(apple.head())
print(apple.tail())
print(apple.isna().sum())


#Missing dates are weekends and market holidays

### Connect Python to AWS RDS Server

In [ ]:
import os
import mysql.connector
import pandas as pd

# Read database credentials from environment variables
connection = mysql.connector.connect(
    host=os.environ["AWS_DB_HOST"],
    user=os.environ["AWS_DB_USER"],
    password=os.environ["AWS_DB_PASSWORD"],
    port=3306,
    connection_timeout=20
)

cursor = connection.cursor()

print("Connected to AWS RDS!")

### Create database 


In [ ]:
cursor.execute("CREATE DATABASE IF NOT EXISTS stock_attention")
cursor.execute("USE stock_attention")

print("Using stock_attention database")

### Create on table for each comapny

In [ ]:
tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

for ticker in tickers:
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS `{ticker}` (
            stock_date DATE PRIMARY KEY,
            open_price DOUBLE,
            high_price DOUBLE,
            low_price DOUBLE,
            close_price DOUBLE,
            adjusted_close DOUBLE,
            volume BIGINT
        )
    """)

print("Five stock tables created")

### Read local CSV files and insert the data

In [ ]:
for ticker in tickers:
    stock = pd.read_csv(data_folder / f"{ticker}.csv")

    # Check the data before inserting it.
    if stock.isna().any().any():
        raise ValueError(f"{ticker} contains missing values")

    if stock["Date"].duplicated().any():
        raise ValueError(f"{ticker} contains duplicate dates")

    # Convert each row into values MySQL can accept.
    rows = []

    for _, row in stock.iterrows():
        rows.append((
            row["Date"],
            float(row["Open"]),
            float(row["High"]),
            float(row["Low"]),
            float(row["Close"]),
            float(row["Adj Close"]),
            int(row["Volume"])
        ))

    insert_sql = f"""
        INSERT INTO `{ticker}` (
            stock_date,
            open_price,
            high_price,
            low_price,
            close_price,
            adjusted_close,
            volume
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            open_price = VALUES(open_price),
            high_price = VALUES(high_price),
            low_price = VALUES(low_price),
            close_price = VALUES(close_price),
            adjusted_close = VALUES(adjusted_close),
            volume = VALUES(volume)
    """

    cursor.executemany(insert_sql, rows)
    connection.commit()

    print(f"{ticker}: loaded {len(rows)} records")

### Verify records inside RDS

In [ ]:
for ticker in tickers:
    cursor.execute(f"""
        SELECT COUNT(*), MIN(stock_date), MAX(stock_date)
        FROM `{ticker}`
    """)

    count, first_date, last_date = cursor.fetchone()

    print(
        ticker,
        "| Records:", count,
        "| First date:", first_date,
        "| Last date:", last_date
    )

### Month with the highest total trading volume for each company

In [ ]:
highest_volume_results = []

for ticker in tickers:
    cursor.execute(f"""
        SELECT
            YEAR(stock_date) AS year,
            MONTH(stock_date) AS month,
            SUM(volume) AS monthly_volume
        FROM `{ticker}`
        WHERE stock_date >= '2020-01-01'
          AND stock_date < '2023-01-01'
        GROUP BY YEAR(stock_date), MONTH(stock_date)
        ORDER BY monthly_volume DESC
    """)

    monthly_results = cursor.fetchall()
    highest_volume = monthly_results[0][2]

    for year, month, monthly_volume in monthly_results:
        if monthly_volume == highest_volume:
            highest_volume_results.append({
                "Ticker": ticker,
                "Year": year,
                "Month": month,
                "Monthly_Volume": int(monthly_volume)
            })

results = pd.DataFrame(highest_volume_results)

print(results.to_string(index=False))

In [ ]:
cursor.close()
connection.close()

print("Connection closed")